In [ ]:
import re
import json
import bibtexparser

# Load notebook
nb_path = "referncestest.ipynb"

with open(nb_path, encoding="utf-8") as f:
    nb = json.load(f)

# Load bib file
with open("ref.bib") as bibfile:
    bib_db = bibtexparser.load(bibfile)

bib_dict = {entry["ID"]: entry for entry in bib_db.entries}

pattern = re.compile(r"\[@([^\]]+)\]")

# --- collect citation order ---
citation_order = []

for cell in nb["cells"]:
    if cell["cell_type"] == "markdown":
        for match in pattern.findall("".join(cell["source"])):
            key = match.strip()
            if key not in citation_order:
                citation_order.append(key)

cite_map = {k: i+1 for i, k in enumerate(citation_order)}

# --- replace inside markdown cells ---
for cell in nb["cells"]:
    if cell["cell_type"] == "markdown":
        text = "".join(cell["source"])

        def repl(match):
            key = match.group(1).strip()
            return f"[{cite_map.get(key, '?')}]"

        new_text = pattern.sub(repl, text)
        cell["source"] = [new_text]

# --- save notebook ---
with open(nb_path, "w", encoding="utf-8") as f:
    json.dump(nb, f, indent=1)

print("✅ Markdown cells updated with numbered citations.")

hello [1]

also also [2]